# Phase C: Full Training Run

Full-budget training with best config from Phase B diagnostic sweep.
Update the config below with the winning Phase 2 ramp settings.


## 1. Environment Setup


In [ ]:
# Install dependencies
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: No GPU detected')


## 1b. Copy GLADA Package to Local Runtime


In [ ]:
import shutil
import os
import sys

# Define paths
DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada"
LOCAL_PKG = '/content/glada'

# Remove existing local package if it exists to ensure a fresh copy
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)

# Copy the directory
try:
    shutil.copytree(DRIVE_PKG, LOCAL_PKG)
    print(f'Successfully copied package to {LOCAL_PKG}')
except Exception as e:
    print(f'Error copying package: {e}')

# Add to sys.path
if LOCAL_PKG not in sys.path:
    sys.path.insert(0, LOCAL_PKG)

# Verify
print(f'Package copied to {LOCAL_PKG}')
print(f"Contents of {LOCAL_PKG}: {os.listdir(LOCAL_PKG)}")


### 1.1 Dataset Extraction

Datasets are stored as compressed archives on Google Drive. To avoid maxing out the
Colab runtime SSD (~80 GB), we process **one archive at a time**:

1. **Copy** the compressed archive from Drive to the runtime SSD
2. **Decompress** from the local copy (much faster than decompressing directly from Drive)
3. **Delete** the archive from the SSD before processing the next dataset

By the time training starts, the runtime holds only decompressed datasets — no archives.

| Dataset | Archive | Role | Reference |
|---|---|---|---|
| ASVspoof 2019 LA (train) | `LA.tar` | Source domain | Wang et al., Computer Speech & Language, 2020 |
| In-the-Wild (ITW) | `release_in_the_wild.tar` | Target domain + eval | Muller et al., Interspeech 2022 |
| MUSAN | `musan.tar.gz` | Additive noise augmentation | Snyder et al., arXiv 2015 (openslr.org/17) |
| RIRS_NOISES | `rirs_noises.zip` | Room impulse responses | Ko et al., Interspeech 2017 (openslr.org/28) |

In [ ]:
import os
import shutil
import tarfile
import zipfile

DRIVE_DATASETS = '/content/drive/MyDrive/Jakob/Datasets'
LOCAL_DATA = '/content/data'


def extract_one_dataset(drive_archive, extract_to, verify_path, name):
    """Copy archive from Drive -> runtime SSD, extract, delete archive.

    Processes one dataset at a time so the runtime SSD never holds
    more than one compressed archive alongside its decompressed data.
    """
    # Skip if already extracted (e.g., runtime reconnect without wipe)
    if os.path.isdir(verify_path) and os.listdir(verify_path):
        print(f'  [{name}] Already extracted at {verify_path}')
        return

    archive_name = os.path.basename(drive_archive)
    local_archive = f'/content/{archive_name}'

    # Step 1: Copy archive to runtime SSD
    print(f'  [{name}] Copying {archive_name} to runtime SSD...')
    shutil.copy2(drive_archive, local_archive)
    size_gb = os.path.getsize(local_archive) / (1024 ** 3)
    print(f'  [{name}] Copied ({size_gb:.2f} GB)')

    # Step 2: Extract locally (faster than extracting from Drive over FUSE)
    os.makedirs(extract_to, exist_ok=True)
    print(f'  [{name}] Extracting to {extract_to}...')
    if archive_name.endswith('.zip'):
        with zipfile.ZipFile(local_archive, 'r') as zf:
            zf.extractall(extract_to)
    elif archive_name.endswith('.tar.gz') or archive_name.endswith('.tgz'):
        with tarfile.open(local_archive, 'r:gz') as tf:
            tf.extractall(extract_to)
    elif archive_name.endswith('.tar'):
        with tarfile.open(local_archive, 'r:') as tf:
            tf.extractall(extract_to)
    else:
        raise ValueError(f'Unknown archive format: {archive_name}')

    # Step 3: Delete archive from SSD (free space before next dataset)
    os.remove(local_archive)
    print(f'  [{name}] Done. Archive deleted from SSD.\n')


# --- Process each dataset one at a time ---
datasets = [
    {
        'name': 'ASVspoof 2019 LA',
        'drive_archive': f'{DRIVE_DATASETS}/LA.tar',
        'extract_to': f'{LOCAL_DATA}/asvspoof',
        'verify_path': f'{LOCAL_DATA}/asvspoof/LA',
    },
    {
        'name': 'In-the-Wild',
        'drive_archive': f'{DRIVE_DATASETS}/release_in_the_wild.tar',
        'extract_to': f'{LOCAL_DATA}/itw',
        'verify_path': f'{LOCAL_DATA}/itw/release_in_the_wild',
    },
    {
        'name': 'MUSAN',
        'drive_archive': f'{DRIVE_DATASETS}/musan.tar.gz',
        'extract_to': LOCAL_DATA,
        'verify_path': f'{LOCAL_DATA}/musan',
    },
    {
        'name': 'RIRS_NOISES',
        'drive_archive': f'{DRIVE_DATASETS}/rirs_noises.zip',
        'extract_to': LOCAL_DATA,
        'verify_path': f'{LOCAL_DATA}/RIRS_NOISES',
    },
]

print('Extracting datasets (one at a time to conserve SSD space):\n')
for ds in datasets:
    assert os.path.exists(ds['drive_archive']), \
        f"Archive not found on Drive: {ds['drive_archive']}"
    extract_one_dataset(**ds)

# --- Define paths used by later cells ---
MUSAN_PATH = '/content/data/musan'
RIR_PATH = '/content/data/RIRS_NOISES/simulated_rirs'

ASV_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO = '/content/data/itw/release_in_the_wild'

# Verify all critical paths exist
for p in [ASV_PROTO, ASV_AUDIO, ITW_META, MUSAN_PATH, RIR_PATH]:
    assert os.path.exists(p), f'Missing: {p}'

# Confirm no leftover archives on runtime SSD
leftover = [f for f in os.listdir('/content/')
            if f.endswith(('.tar', '.tar.gz', '.tgz', '.zip'))
            and os.path.isfile(f'/content/{f}')]
assert not leftover, f'Leftover archives on SSD (should be deleted): {leftover}'

print('All datasets extracted and verified. No compressed archives on SSD.')

In [ ]:
# Install glada package from local copy (copied from Drive in cell above)
# The package was copied to /content/glada_code/ which contains pyproject.toml + glada/
import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/glada_code'])
# Core configuration and utilities
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed

# Dataset classes
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,  # Source domain (labeled + speaker IDs)
    ASVspoofEvalDataset,          # In-domain eval (ASVspoof 2019 LA eval set)
    InTheWildDataset,             # Target domain (unlabeled for DA) + eval (labeled)
    create_dataloaders,           # DataLoader factory with balanced sampling
)

# Augmentation pipeline: RawBoost + MUSAN + RIR (Huang et al., IB-CAAN 2025)
from glada.augmentation import RawBoost, CombinedAugmenter

# Trainer: two-phase training loop with GRL scheduling
from glada.trainers import GLADATrainer

# Evaluation: EER with bootstrap CI + checkpoint averaging
from glada.evaluation import (
    MultiDatasetEvaluator,
    average_checkpoints,
    save_averaged_checkpoint,
)

import torch
import json

print(f'GLADA package loaded.')
print(f'Available backbones: {list(BACKBONE_REGISTRY.keys())}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')

## 2. Configuration

GLADA uses a centralized `GLADAConfig` dataclass (see `glada/config.py`) with:
- **Backbone selection** via `BACKBONE_REGISTRY` (WavLM-Large, XLS-R, etc.)
- **Ablation switches** to toggle any component on/off
- **Hyperparameters** tuned based on prior work and stability experiments

### Key Hyperparameter Choices

| Parameter | Value | Rationale |
|---|---|---|
| `domain_loss_weight` ($\alpha$) | 0.1 | >0.1 destabilizes training (empirically validated) |
| `vib_beta` ($\beta$) | 1e-4 | Alemi et al. 2017; higher causes KL explosion |
| `speaker_loss_weight` ($\gamma$) | 0.1 | Max value after warmup |
| `cka_loss_weight` ($\delta$) | 0.01 | Tran et al. 2025 |
| `grl_warmup_epochs` | 5 | Ganin et al. 2016 — <5 causes instability |
| `llrd_decay` | 0.95 | Clark et al. 2020 — standard for SSL fine-tuning |
| `batch_size` | 24 | Per-domain; fits T4 (16 GB) and A100 (40/80 GB) |
| `effective_batch_size` | 384 | Via gradient accumulation (24 x 2 domains x 8 accum) |

### Augmentation Pipeline (Huang et al., IB-CAAN 2025)

The full augmentation pipeline chains three strategies, each firing independently:

| Stage | Reference | Probability | Effect |
|---|---|---|---|
| RawBoost | Tak et al., ICASSP 2022 | 0.5 | Codec/channel artifacts |
| MUSAN | Snyder et al., 2015 | 0.5 | Additive noise (speech/music/noise) |
| RIR | Ko et al., Interspeech 2017 | 0.5 | Room reverberation |

IB-CAAN achieved 4.61% ITW EER with all three, vs 4.93% with RawBoost alone.

### Checkpointing

Checkpoints saved every epoch to enable top-K checkpoint averaging post-training
(Huang et al. averaged top-5 by EER for an additional ~0.3% improvement).

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# Modify these values to run different experiments.
# ============================================================

BACKBONE = 'wavlm-large'   # Options: wavlm-large, xlsr-300m, wavlm-base-plus, etc.
SEED = 42                   # Reproducibility seed (run 42, 123, 456 for multi-seed)

config = GLADAConfig(
    experiment_name=f'glada_{BACKBONE}_seed{SEED}',
    backbone=BACKBONE,
    seed=SEED,

    # --- Training ---
    phase1_epochs=5,           # Frozen backbone warmup
    phase2_epochs=45,          # End-to-end fine-tuning with LLRD
    batch_size=64,             # Per-domain batch size
    effective_batch_size=384,  # Gradient accumulation target
    head_lr=1e-3,              # Phase 1 + head params in Phase 2
    backbone_lr=1e-5,          # Phase 2 peak (scaled by LLRD per layer)

    # --- Loss Weights ---
    domain_loss_weight=0.1,    # alpha: GLADA per-group DA weight (<=0.1)
    vib_beta=1e-4,             # beta: VIB KL divergence weight
    speaker_loss_weight=0.1,   # gamma: Speaker adversary max weight (warmed up)
    cka_loss_weight=0.01,      # delta: CKA group diversity weight

    # --- Augmentation (RawBoost + MUSAN + RIR) ---
    rawboost_prob=0.5,         # P(apply RawBoost to source sample)
    musan_path=MUSAN_PATH,     # MUSAN corpus root (Snyder et al., 2015)
    musan_prob=0.5,            # P(apply MUSAN noise)
    rir_path=RIR_PATH,         # RIR corpus root (Ko et al., 2017)
    rir_prob=0.5,              # P(apply RIR convolution)

    # --- Ablation Switches (all True for full GLADA) ---
    use_glada=True,            # Per-group domain discriminators (NOVEL)
    use_vib=True,              # Variational Information Bottleneck
    use_speaker_adversary=True,# Speaker disentanglement via GRL
    use_cka_loss=True,         # CKA diversity regularization
    use_hierarchical=True,     # Hierarchical grouped-layer pooling
    use_rawboost=True,         # RawBoost augmentation

    # --- Checkpointing (save every epoch for top-K averaging) ---
    save_every_n_epochs=1,     # Save every epoch (Huang et al. 2025 averaged top-5)
    save_top_k=10,             # Keep 10 most recent periodic checkpoints
)

# --- Derived values (computed in __post_init__) ---
print(f'Backbone: {config.backbone} ({config.hf_model_id})')
print(f'  Layers: {config.num_layers}, Groups: {config.num_groups}, Dim: {config.hidden_dim}')
print(f'  Grad accum: {config.grad_accum_steps} '
      f'(batch {config.batch_size} x 2 domains x {config.grad_accum_steps} = {config.effective_batch_size})')
print(f'  Total epochs: {config.total_epochs} '
      f'(Phase 1: {config.phase1_epochs}, Phase 2: {config.phase2_epochs})')
print(f'  Augmentation: MUSAN={"ON" if config.musan_path else "OFF"}, '
      f'RIR={"ON" if config.rir_path else "OFF"}')

## 3. Dataset Loading

### Domain Adaptation Setup

GLADA uses **unsupervised domain adaptation** (Ganin et al. 2016):
- **Source domain** (ASVspoof 2019 LA train): Labeled data with task labels + speaker IDs
- **Target domain** (ITW): Unlabeled — only domain membership (source=0, target=1) is used

The GRL reverses gradients from the domain discriminator, forcing the feature extractor
to learn representations that are indistinguishable between ASVspoof and ITW — i.e.,
domain-invariant features that generalize across datasets.

### Balanced Sampling

ASVspoof 2019 LA has significantly more spoof samples than bonafide. We use
`WeightedRandomSampler` with inverse class frequency weights to ensure ~50/50
class balance in each batch, preventing the classifier from defaulting to the majority class.

In [ ]:
# --- Create Datasets ---
# Source domain: ASVspoof 2019 LA training set (labeled + speaker IDs)
# Augmentation pipeline: RawBoost + MUSAN + RIR via CombinedAugmenter.
# CombinedAugmenter reads musan_path, rir_path, and per-stage probabilities
# from config, initializing only the stages whose paths exist.
print('Loading datasets...')

# Build combined augmentation pipeline (Huang et al., IB-CAAN 2025)
augmenter = CombinedAugmenter(config)
print(f'Augmenter: {augmenter}')

source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=config,
    augmenter=augmenter,
)

# Target domain: ITW (unlabeled for adversarial adaptation)
# use_labels=False: only domain membership used by GRL, not task labels
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config,
    use_labels=False,
)

# Evaluation: ITW with ground truth labels for EER computation
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config,
    use_labels=True,
)

# Optional: ASVspoof 2019 LA eval set (in-domain evaluation)
extra_eval = {}
if os.path.exists(ASV_EVAL_PROTO) and os.path.exists(ASV_EVAL_AUDIO):
    extra_eval['2019_LA'] = ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO,
        audio_dir=ASV_EVAL_AUDIO,
        config=config,
    )

print(f'\nSource: {len(source_ds):,} samples, {source_ds.num_speakers} speakers')
print(f'Target: {len(target_ds):,} samples (unlabeled)')
print(f'Eval:   {len(eval_ds):,} samples (ITW, labeled)')
for name, ds in extra_eval.items():
    print(f'Extra:  {len(ds):,} samples ({name})')

In [ ]:
# --- Create DataLoaders ---
# Uses balanced sampling for source domain (inverse class frequency weighting)
# drop_last=True for training to ensure consistent batch sizes with grad accumulation

loaders = create_dataloaders(
    config=config,
    source_ds=source_ds,
    target_ds=target_ds,
    eval_ds=eval_ds,
    extra_eval=extra_eval,
)

source_loader = loaders['source']
target_loader = loaders['target']
eval_loader = loaders['eval']
extra_eval_loaders = {k: v for k, v in loaders.items()
                      if k not in ('source', 'target', 'eval')}

print(f'\nSource loader: {len(source_loader)} batches (batch_size={config.batch_size})')
print(f'Target loader: {len(target_loader)} batches')
print(f'Eval loader:   {len(eval_loader)} batches (batch_size={config.batch_size * 4})')

## 4. Training

### Two-Phase Training Strategy

**Phase 1** (5 epochs): Freeze SSL backbone, train only the randomly-initialized heads
(classifier, discriminators, speaker adversary, pooling, VIB). This stabilizes the
heads before the backbone is unfrozen.

**Phase 2** (45 epochs): Unfreeze backbone with gradient checkpointing (to fit in GPU
memory) and LLRD (Clark et al. 2020). Lower SSL layers get smaller learning rates
to preserve pre-trained acoustic knowledge; higher layers adapt more aggressively.

$$\text{lr}_i = \text{backbone\_lr} \times \text{decay}^{(L - i)}$$

where $L$ is the max layer index and $i$ is the current layer. With `decay=0.95` and
`backbone_lr=1e-5`, layer 0 gets ~2.9e-7 while layer 24 gets the full 1e-5.

### GRL Alpha Scheduling (Ganin et al. 2016, Section 3.3)

The GRL reversal strength $\alpha$ is warmed up linearly from 0 to 1 over
`grl_warmup_epochs` (5). Starting with $\alpha=0$ means no domain adaptation
initially — the heads learn basic classification first, then domain invariance
is gradually enforced.

### Checkpointing

Checkpoints are saved to Google Drive after every epoch. The best model (lowest ITW EER)
is saved as `best_model.pt`. Training can resume from any checkpoint if Colab disconnects.

In [ ]:
# ============================================================
# TRAIN GLADA
# Set RESUME_FROM to a checkpoint path to resume after Colab disconnection.
# ============================================================

RESUME_FROM = None  # e.g., '/content/drive/MyDrive/Jakob/Experiments/.../best_model.pt'

trainer = GLADATrainer(
    config=config,
    source_loader=source_loader,
    target_loader=target_loader,
    eval_loader=eval_loader,
    extra_eval_loaders=extra_eval_loaders,
    num_speakers=source_ds.num_speakers,
    resume_from=RESUME_FROM,
)

results = trainer.train()

## 5. Multi-Dataset Evaluation with Bootstrap CI

After training, we evaluate the best checkpoint on all available datasets and compute
**bootstrap 95% confidence intervals** for the EER. This is critical for thesis defense
(Rule 3) — the committee requires statistical significance reporting.

### Equal Error Rate (EER)

EER is the operating point where the false acceptance rate equals the false rejection rate:

$$\text{EER} = \text{FAR}(\tau^*) = \text{FRR}(\tau^*) \quad \text{where} \quad \tau^* = \arg\min_\tau |\text{FAR}(\tau) - \text{FRR}(\tau)|$$

Lower EER = better detection. Current SOTA on ITW (Feb 2026): ~1.7-4.4%.

### Bootstrap Confidence Intervals

We resample the test set with replacement 1000 times, compute EER on each resample,
and report the 2.5th and 97.5th percentiles as the 95% CI.

In [ ]:
# --- Load best checkpoint and evaluate ---
from glada.models import GLADAModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best model
best_path = os.path.join(results['run_dir'], 'best_model.pt')
model = GLADAModel(config)
ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
model.to(device)
model.eval()

print(f'Loaded best model from epoch {results["best_epoch"]} '
      f'(ITW EER: {results["best_eer"]:.2f}%)')

# Build eval loaders dict
eval_loaders_dict = {'ITW': eval_loader}
eval_loaders_dict.update(extra_eval_loaders)

# Run evaluation with bootstrap CI
evaluator = MultiDatasetEvaluator(model, config, device, eval_loaders_dict)
all_results = evaluator.evaluate_all()

# Print formatted results
print('\n' + evaluator.results_to_markdown(all_results))
print('\nLaTeX row:')
print(evaluator.results_to_latex(all_results))

In [ ]:
# --- Save evaluation results to JSON ---
results_path = os.path.join(results['run_dir'], 'eval_results.json')

# Convert numpy types to Python types for JSON serialization
serializable = {}
for name, r in all_results.items():
    serializable[name] = {
        k: float(v) if hasattr(v, 'item') else v
        for k, v in r.items()
    }

with open(results_path, 'w') as f:
    json.dump(serializable, f, indent=2)
print(f'Results saved to {results_path}')

## 5.1 Checkpoint Averaging (Polyak & Juditsky, 1992; Huang et al., 2025)

Averaging the weights of the top-K checkpoints (by validation EER) produces a model
whose parameters lie closer to the center of the loss basin, which tends to be flatter
and thus generalizes better (Izmailov et al., 2018, SWA).

IB-CAAN (Huang et al., 2025) improved ITW EER from 4.93% to 4.61% by averaging
their top-5 checkpoints — a meaningful gain at zero additional training cost.

$$\theta_{\text{avg}} = \frac{1}{K} \sum_{i=1}^{K} \theta_i$$

where $\theta_i$ are the parameters of the $i$-th best checkpoint by EER.

In [ ]:
# --- Top-5 Checkpoint Averaging ---
# Requires save_every_n_epochs >= 1 during training (set above).
# Averages the top-5 checkpoints by ITW EER, then evaluates the averaged model.

from glada.models import GLADAModel

TOP_K = 5

try:
    avg_model = average_checkpoints(
        run_dir=results['run_dir'],
        model_class=GLADAModel,
        config=config,
        top_k=TOP_K,
        metric='eer',
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )

    # Evaluate averaged model
    eval_loaders_dict_avg = {'ITW': eval_loader}
    eval_loaders_dict_avg.update(extra_eval_loaders)

    evaluator_avg = MultiDatasetEvaluator(avg_model, config, device, eval_loaders_dict_avg)
    avg_results = evaluator_avg.evaluate_all()

    print(f'\n--- Checkpoint Averaged (top-{TOP_K}) Results ---')
    print(evaluator_avg.results_to_markdown(avg_results))

    # Compare with single best
    best_itw = results['best_eer']
    avg_itw = avg_results['ITW']['eer']
    delta = best_itw - avg_itw
    print(f'Single best: {best_itw:.2f}% | Averaged: {avg_itw:.2f}% | Delta: {delta:+.2f}%')

    # Save averaged checkpoint
    save_averaged_checkpoint(
        avg_model, results['run_dir'], config, TOP_K,
        source_epochs=[],  # epochs logged by average_checkpoints
    )

except Exception as e:
    print(f'Checkpoint averaging failed: {e}')
    print('This is expected if save_every_n_epochs was 0 during training.')


## 6. Training Visualization

Six subplots for thesis figures:
1. **Total loss** over epochs
2. **Loss components** (task, domain, speaker) — shows relative contributions
3. **KL + CKA losses** — VIB health monitoring (KL should stabilize, not explode)
4. **EER per dataset** — generalization across domains
5. **AUC per dataset** — complementary to EER
6. **Task + speaker accuracy** — task acc should increase, speaker acc should decrease
   (decreasing speaker acc means GRL is successfully removing speaker-specific info)

Vertical dashed line marks the Phase 1 -> Phase 2 transition.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = results['history']
epochs = [h['epoch'] for h in history]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'GLADA Training: {config.backbone} (seed={config.seed})', fontsize=14)

# --- Plot 1: Total Loss ---
ax = axes[0, 0]
ax.plot(epochs, [h['train']['total_loss'] for h in history], 'b-')
ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
ax.set_title('Total Training Loss')
ax.axvline(x=config.phase1_epochs, color='r', linestyle='--', alpha=0.5, label='Phase 2')
ax.legend()

# --- Plot 2: Loss Components ---
ax = axes[0, 1]
ax.plot(epochs, [h['train']['spoof_loss'] for h in history], label='Task (CE)')
ax.plot(epochs, [h['train']['glada_loss'] for h in history], label='GLADA (DA)')
ax.plot(epochs, [h['train']['speaker_loss'] for h in history], label='Speaker')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Loss Components')
ax.axvline(x=config.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.legend()

# --- Plot 3: KL + CKA ---
ax = axes[0, 2]
ax.plot(epochs, [h['train']['kl_loss'] for h in history], label='KL (clamped)')
if any(h['train'].get('kl_raw', 0) > 0 for h in history):
    ax.plot(epochs, [h['train'].get('kl_raw', 0) for h in history],
            label='KL (raw)', alpha=0.5)
ax.plot(epochs, [h['train']['cka_loss'] for h in history], label='CKA')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('KL + CKA Losses')
ax.axvline(x=config.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.legend()

# --- Plot 4: EER per Dataset ---
ax = axes[1, 0]
dataset_names = list(history[0]['eval'].keys()) if history else []
for ds_name in dataset_names:
    eers = [h['eval'][ds_name]['eer'] for h in history if ds_name in h['eval']]
    ax.plot(epochs[:len(eers)], eers, label=ds_name)
ax.set_xlabel('Epoch'); ax.set_ylabel('EER (%)')
ax.set_title('EER per Dataset')
ax.axvline(x=config.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.legend()

# --- Plot 5: AUC per Dataset ---
ax = axes[1, 1]
for ds_name in dataset_names:
    aucs = [h['eval'][ds_name]['auc'] for h in history if ds_name in h['eval']]
    ax.plot(epochs[:len(aucs)], aucs, label=ds_name)
ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
ax.set_title('AUC per Dataset')
ax.axvline(x=config.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.legend()

# --- Plot 6: Task + Speaker Accuracy ---
ax = axes[1, 2]
ax.plot(epochs, [h['train']['spoof_acc'] * 100 for h in history], label='Task Acc')
ax.plot(epochs, [h['train']['speaker_acc'] * 100 for h in history], label='Speaker Acc')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Task + Speaker Accuracy')
ax.axvline(x=config.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.legend()

plt.tight_layout()

# Save figure to run directory for thesis
fig_path = os.path.join(results['run_dir'], 'training_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figure saved to {fig_path}')
plt.show()

## 7. Summary

In [ ]:
print('=' * 60)
print(f'EXPERIMENT COMPLETE: {config.experiment_name}')
print(f'  Backbone: {config.backbone}')
print(f'  Best ITW EER: {results["best_eer"]:.2f}% (epoch {results["best_epoch"]})')
print(f'  Run dir: {results["run_dir"]}')
print(f'\nCheckpoints saved:')
print(f'  best_model.pt       -- best ITW EER checkpoint')
print(f'  config.json         -- experiment configuration')
print(f'  history.json        -- full training history')
print(f'  eval_results.json   -- final evaluation with bootstrap CI')
print(f'  training_curves.png -- visualization')
print('=' * 60)
print(f'\nNext steps:')
print(f'  1. Run ablation experiments (glada_ablation.ipynb)')
print(f'  2. Run multi-dataset evaluation (glada_evaluation.ipynb)')
print(f'  3. Generate thesis figures (glada_analysis.ipynb)')

---

## Full Reference List

1. **Alemi et al.** "Deep Variational Information Bottleneck", ICLR 2017 — VIB objective
2. **Clark et al.** "Electra: Pre-training Text Encoders as Discriminators", EMNLP 2020 — LLRD
3. **El-Kheir et al.** "Layer-wise Analysis of Self-Supervised Speech Models for Deepfake Detection", NAACL 2025 — Per-layer norm motivation
4. **Fang et al.** "Multi-Layer Adversarial Domain Adaptation", Neurocomputing 2021 — MLADA (vision precedent for GLADA)
5. **Ganin et al.** "Domain-Adversarial Training of Neural Networks", JMLR 2016 — GRL and DANN
6. **Huang et al.** "IB-CAAN", 2025 — VIB for deepfake detection
7. **Izmailov et al.** "Averaging Weights Leads to Wider Optima and Better Generalization", UAI 2018 — SWA
8. **Ko et al.** "A Study on Data Augmentation of Reverberant Speech", Interspeech 2017 — RIR augmentation
9. **Kornblith et al.** "Similarity of Neural Network Representations Revisited", ICML 2019 — CKA
10. **Liang et al.** "HierCon", WWW 2026 — Grouped layer attention
11. **Muller et al.** "Does Audio Deepfake Detection Generalize?", Interspeech 2022 — ITW dataset
12. **Ngo Trung et al.** "Domain Adaptation with Adapter", ACL-IJCNLP 2021 — DAA (NLP precedent)
13. **Okabe et al.** "Attentive Statistics Pooling", Interspeech 2018 — ASP
14. **Pan et al.** "Attentive Merging of Hidden Embeddings", Interspeech 2024
15. **Polyak & Juditsky** "Acceleration of Stochastic Approximation by Averaging", SIAM 1992 — Checkpoint averaging theory
16. **Smith & Topin** "Super-Convergence", 2019 — OneCycleLR
17. **Snyder et al.** "MUSAN: A Music, Speech, and Noise Corpus", arXiv 2015 — MUSAN noise augmentation
18. **Tak et al.** "RawBoost", ICASSP 2022 — Raw waveform augmentation
19. **Tran et al.** "Multi-scale Convolutional Representations", ACM MM 2025 — CKA diversity
20. **Wang et al.** "ASVspoof 2019", Computer Speech & Language, 2020 — Dataset
